In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
df = spark.read.csv("/Volumes/demo/default/demo/C1.csv",header=True,inferSchema=True)
#df=spark.read.option("header","true").option("inferSchrema","true").csv("/Volumes/demo/default/demo/C1.csv")
df.show()


+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     blank|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:

df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,blank,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,Blank
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,blank,450,C


In [0]:
#Handle blank values
from pyspark.sql.functions import *
df=df.replace(["blank","Blank"],None)
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


In [0]:
#Make upper ot lower
df=df.withColumn("Country",upper(col("Country")))
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   INDIA|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023| NULL|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|    NULL|
|       106|Mallory|NEW YORK|03-15-2023|  400|       B|
|       107|  Trent|   INDIA|10-04-2023|  350|       B|
|       108|    Bob|   INDIA|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|      NULL|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:
#Fix Date format
df=df.withColumn("JoinDate", try_to_date(col("JoinDate"),"dd-mm-yyyy"))
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|2023-01-15|  250|       A|
|       102|    Bob|   INDIA|2023-01-01|  150|       B|
|       103|Charlie|      UK|2023-01-20| NULL|       C|
|       104|  Alice|     USA|2023-01-15|  250|       A|
|       105|    Eve|      UK|2023-01-01|  300|    NULL|
|       106|Mallory|NEW YORK|2023-01-03|  400|       B|
|       107|  Trent|   INDIA|2023-01-10|  350|       B|
|       108|    Bob|   INDIA|2023-01-05|  150|       B|
|       109|  Oscar|     USA|2023-01-28|  500|       A|
|       110|  Peggy|      UK|      NULL|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:
#Fill values for missing vales
df=df.fillna({
    "Category" : "Unknown",
    "Sales" : 0
})
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|2023-01-15|  250|       A|
|       102|    Bob|   INDIA|2023-01-01|  150|       B|
|       103|Charlie|      UK|2023-01-20|    0|       C|
|       104|  Alice|     USA|2023-01-15|  250|       A|
|       105|    Eve|      UK|2023-01-01|  300| Unknown|
|       106|Mallory|NEW YORK|2023-01-03|  400|       B|
|       107|  Trent|   INDIA|2023-01-10|  350|       B|
|       108|    Bob|   INDIA|2023-01-05|  150|       B|
|       109|  Oscar|     USA|2023-01-28|  500|       A|
|       110|  Peggy|      UK|      NULL|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:
data = [
    (1, "Laptop", 50000, 2, "Electronics"),
    (2, "Mobile", 20000, 3, "Electronics"),
    (3, "Tablet", 15000, 1, "Electronics"),
    (4, "Headphones", 2000, 5, "Accessories"),
    (5, "Keyboard", 1500, 4, "Accessories"),
    (6, "Laptop", 52000, None, "Electronics"),   # null quantity
    (7, "", 18000, 2, "Electronics"),            # blank product
    (8, "Mouse", None, 3, "Accessories"),        # null price
    (9, "Monitor", 12000, -1, "Electronics"),    # invalid quantity
    (10, "Laptop", 50000, 2, "Electronics")      # duplicate-like row
]

columns = ["product_id", "product_name", "price", "quantity", "category"]

sales = spark.createDataFrame(data, columns)
sales.show()

+----------+------------+-----+--------+-----------+
|product_id|product_name|price|quantity|   category|
+----------+------------+-----+--------+-----------+
|         1|      Laptop|50000|       2|Electronics|
|         2|      Mobile|20000|       3|Electronics|
|         3|      Tablet|15000|       1|Electronics|
|         4|  Headphones| 2000|       5|Accessories|
|         5|    Keyboard| 1500|       4|Accessories|
|         6|      Laptop|52000|    NULL|Electronics|
|         7|            |18000|       2|Electronics|
|         8|       Mouse| NULL|       3|Accessories|
|         9|     Monitor|12000|      -1|Electronics|
|        10|      Laptop|50000|       2|Electronics|
+----------+------------+-----+--------+-----------+



In [0]:
sales=sales.replace("","Unknown")
sales=sales.fillna({
    "price":0,7
    "quantity":0,
    "category":"Unknown"
})
sales.show()

+----------+------------+-----+--------+-----------+
|product_id|product_name|price|quantity|   category|
+----------+------------+-----+--------+-----------+
|         1|      Laptop|50000|       2|Electronics|
|         2|      Mobile|20000|       3|Electronics|
|         3|      Tablet|15000|       1|Electronics|
|         4|  Headphones| 2000|       5|Accessories|
|         5|    Keyboard| 1500|       4|Accessories|
|         6|      Laptop|52000|       0|Electronics|
|         7|     Unknown|18000|       2|Electronics|
|         8|       Mouse|    0|       3|Accessories|
|         9|     Monitor|12000|      -1|Electronics|
|        10|      Laptop|50000|       2|Electronics|
+----------+------------+-----+--------+-----------+

